# AdaptAuth-ML · Experimento S8, versión 4 (corrección según diagnóstico)

**Desde el celular:** **Entorno de ejecución → Ejecutar todo** y acepta el permiso de Google Drive. Como la muestra ya está guardada en tu Drive, tarda unos **5 a 10 minutos**.

Cambios respecto a la v3, basados en el diagnóstico del RBA real:
- **Se evalúan los inicios de sesión con contraseña correcta**, que es donde actúa la autenticación basada en riesgo. Los intentos fallidos se usan como contexto.
- **Variables de contexto global sin etiquetas:** tipo de dispositivo y tasa previa de fallos y volumen del ASN y del país.
- **La regla P5 (OTP a todo primer acceso) queda desactivada por defecto**, porque generaba casi todo el FPR. También se reporta con P5 activa.
- **Compara 6 modelos en una sola corrida:** reglas estáticas, IF v3, IF con contexto global, AdaptAuth-ML v4 (la propuesta), AdaptAuth-ML v4 con P5 y una referencia supervisada (cota superior; no es la propuesta).

In [ ]:
!pip -q install shap

## 1. Código

In [ ]:
%%writefile adaptauth.py
"""
AdaptAuth-ML - pipeline del baseline S8 (Isolation Forest + reglas contextuales).

Implementa el diseño corregido del PI7 (Fase 0):
  - x_i en R^8 (hora cíclica, desviación horaria, historial, frecuencias de país/ASN/dispositivo, RTT)
  - c_i = 0.25 r1 + 0.25 r2 + 0.30 r3 + 0.15 r4 + 0.05 r5
  - s(x) de Isolation Forest (ecuación 1) y reescalado s'(x) (ecuación 1b)
  - R_i = alpha * s'(x_i) + (1 - alpha) * c_i (ecuación 2)
  - Política: acceso directo / OTP / bloqueo, con regla de arranque en frío (P5)
  - Partición temporal 70/15/15 y calibración de alpha, theta1, theta2 solo en validación

Los nombres de columnas siguen el dataset RBA (Wiefling et al., 2022), de modo que el
mismo código corre sobre el archivo real o sobre los datos sintéticos de prueba.
"""
from __future__ import annotations

import json
import time
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import IsolationForest

# --------------------------------------------------------------------------------------
# Columnas del dataset RBA
# --------------------------------------------------------------------------------------
COL_TS = "Login Timestamp"
COL_USER = "User ID"
COL_RTT = "Round-Trip Time [ms]"
COL_IP = "IP Address"
COL_COUNTRY = "Country"
COL_ASN = "ASN"
COL_BROWSER = "Browser Name and Version"
COL_OS = "OS Name and Version"
COL_DEVICE = "Device Type"
COL_SUCCESS = "Login Successful"
COL_ATTACK_IP = "Is Attack IP"          # SOLO etiqueta de evaluación, nunca entrada
COL_ATO = "Is Account Takeover"         # SOLO etiqueta de evaluación, nunca entrada

LABEL_COLUMNS = (COL_ATTACK_IP, COL_ATO)

FEATURES = [
    "hora_sin", "hora_cos", "desv_horaria", "log_n_prev",
    "frec_pais", "frec_asn", "frec_dispositivo", "log_rtt",
]
# Contexto global (v4): no usa etiquetas, solo el pasado de TODA la población
FEATURES_GLOBAL = ["es_movil", "es_desktop", "tasa_fallo_asn", "tasa_fallo_pais", "log_vol_asn"]
RULES = ["r1_dispositivo_nuevo", "r2_pais_poco_habitual", "r3_viaje_imposible",
         "r4_horario_atipico", "r5_asn_nuevo"]
RULE_WEIGHTS = {"r1_dispositivo_nuevo": 0.25, "r2_pais_poco_habitual": 0.25,
                "r3_viaje_imposible": 0.30, "r4_horario_atipico": 0.15, "r5_asn_nuevo": 0.05}

DECISION_NAMES = {0: "acceso_directo", 1: "otp", 2: "bloqueo"}


@dataclass
class Config:
    seed: int = 42
    n_estimators: int = 100          # t
    max_samples: int = 256           # n
    split: tuple = (0.70, 0.15, 0.15)
    alpha_grid: tuple = (0.3, 0.5, 0.7, 0.85, 1.0)
    theta_grid: tuple = tuple(np.round(np.arange(0.02, 0.951, 0.01), 2))
    max_fpr_theta1: float = 0.05     # RNF02 / CA-C2-02
    max_fpr_theta2: float = 0.01     # bloqueo: tolerancia más estricta
    cold_start_otp: bool = True      # regla P5
    n_test_blocks: int = 10          # bloques temporales para la prueba pareada
    label: str = "anomalo"           # columna de verdad usada para calibrar


# --------------------------------------------------------------------------------------
# 1. Datos
# --------------------------------------------------------------------------------------
def _open_csv(path: str):
    if str(path).endswith(".zip"):          # se lee directo del .zip, sin descomprimir ~9 GB
        import zipfile
        zf = zipfile.ZipFile(path)
        member = next(n for n in zf.namelist() if n.endswith(".csv"))
        return zf.open(member)
    return path


def load_rba(path: str, nrows: int | None = None, user_pct: float | None = None,
             chunksize: int = 2_000_000) -> pd.DataFrame:
    """
    Carga el CSV real de RBA (rba-dataset.csv o el .zip de Zenodo).
      - nrows: primeras N filas (ventana de tiempo corta: muchos usuarios sin historial).
      - user_pct: muestreo POR USUARIO sobre todo el archivo (p. ej. 5 = 5 % de usuarios con
        TODOS sus accesos). Conserva historiales completos; es la opción recomendada.
    """
    usecols = [COL_TS, COL_USER, COL_RTT, COL_IP, COL_COUNTRY, COL_ASN, COL_BROWSER,
               COL_OS, COL_DEVICE, COL_SUCCESS, COL_ATTACK_IP, COL_ATO]
    if str(path).endswith(".parquet"):      # muestra ya filtrada y guardada (lectura en segundos)
        df = pd.read_parquet(path)
        df[COL_TS] = pd.to_datetime(df[COL_TS])
        return df
    if user_pct:
        parts = []
        for ch in pd.read_csv(_open_csv(path), usecols=usecols, chunksize=chunksize):
            h = pd.util.hash_pandas_object(ch[COL_USER], index=False).to_numpy() % 10000
            parts.append(ch[h < user_pct * 100])
        df = pd.concat(parts, ignore_index=True)
    else:
        df = pd.read_csv(_open_csv(path), usecols=usecols, nrows=nrows)
    df[COL_TS] = pd.to_datetime(df[COL_TS])
    for col in (COL_SUCCESS, COL_ATTACK_IP, COL_ATO):
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.lower().isin(["true", "1"])
    return df


def generate_synthetic_rba(n_users: int = 2000, days: int = 365, seed: int = 42) -> pd.DataFrame:
    """
    Datos SINTÉTICOS con la misma estructura de columnas que RBA. Sirven solo para
    probar el pipeline de punta a punta; sus métricas NO son resultados de la tesis.
    """
    rng = np.random.default_rng(seed)
    start = pd.Timestamp("2020-02-03")
    countries = np.array(["NO", "SE", "DE", "GB", "US", "FR", "ES", "PL", "BR", "IN"])
    home_p = np.array([0.55, 0.08, 0.07, 0.06, 0.06, 0.05, 0.04, 0.04, 0.03, 0.02])
    asn_pool = {c: [10000 + 100 * i + k for k in range(6)] for i, c in enumerate(countries)}
    attacker_countries = np.array(["US", "RU", "CN", "VN", "BR", "NL", "UA", "ID"])
    attacker_asns = [64000 + k for k in range(60)]       # proveedores de hosting
    browsers = ["Chrome 88", "Firefox 85", "Safari 14", "Edge 88", "Chrome Mobile 88",
                "Samsung Internet 13", "Opera 74"]
    oses = ["Windows 10", "Mac OS X 10.15", "iOS 14.4", "Android 10", "Linux", "Android 11"]
    devices = ["desktop", "mobile", "tablet"]
    base_rtt = {c: v for c, v in zip(countries, [30, 45, 60, 65, 140, 70, 90, 70, 220, 260])}

    rows = []
    for uid in range(n_users):
        home = rng.choice(countries, p=home_p)
        user_asns = rng.choice(asn_pool[home], size=rng.integers(1, 3), replace=False)
        n_dev = rng.integers(1, 4)
        user_devs = [(rng.choice(browsers), rng.choice(oses), rng.choice(devices, p=[.5, .42, .08]))
                     for _ in range(n_dev)]
        pref_hour = np.clip(rng.normal(13, 3), 6, 22)
        n_events = int(np.clip(rng.lognormal(4.2, 0.9), 2, 900))
        times = np.sort(rng.uniform(0, days, n_events))
        for t in times:
            day = np.floor(t)
            hour = (rng.normal(pref_hour, 2.5)) % 24
            ts = start + pd.Timedelta(days=float(day)) + pd.Timedelta(hours=float(hour))
            ctry, asn = home, rng.choice(user_asns)
            if rng.random() < 0.02:                       # viaje legítimo
                ctry = rng.choice(countries)
                asn = rng.choice(asn_pool[ctry])
            elif rng.random() < 0.08:                     # red móvil / otra red del país
                asn = rng.choice(asn_pool[home])
            dev = user_devs[rng.integers(len(user_devs))]
            if rng.random() < 0.03:                       # dispositivo nuevo legítimo
                dev = (rng.choice(browsers), rng.choice(oses), rng.choice(devices))
                user_devs.append(dev)
            rtt = rng.lognormal(np.log(base_rtt[ctry]), 0.35)
            rows.append((ts, uid, rtt, ctry, asn, dev[0], dev[1], dev[2],
                         rng.random() > 0.04, 0, 0))

    n_legit = len(rows)
    # Relleno de credenciales (credential stuffing) desde IPs de ataque
    n_attack = int(n_legit * 0.08)
    for _ in range(n_attack):
        uid = int(rng.integers(n_users))
        ts = start + pd.Timedelta(days=float(rng.uniform(0, days)))
        ctry = rng.choice(attacker_countries)
        rtt = rng.lognormal(np.log(180), 0.6)
        rows.append((ts, uid, rtt, ctry, rng.choice(attacker_asns), rng.choice(browsers),
                     rng.choice(oses), "desktop", rng.random() < 0.02, 1, 0))
    # Toma de cuenta (account takeover): acceso exitoso del atacante
    n_ato = int(n_legit * 0.002)
    for _ in range(n_ato):
        uid = int(rng.integers(n_users))
        ts = start + pd.Timedelta(days=float(rng.uniform(30, days)))
        ctry = rng.choice(attacker_countries)
        rtt = rng.lognormal(np.log(180), 0.6)
        rows.append((ts, uid, rtt, ctry, rng.choice(attacker_asns), rng.choice(browsers),
                     rng.choice(oses), rng.choice(devices), True, 1, 1))

    df = pd.DataFrame(rows, columns=[COL_TS, COL_USER, COL_RTT, COL_COUNTRY, COL_ASN,
                                     COL_BROWSER, COL_OS, COL_DEVICE, COL_SUCCESS,
                                     COL_ATTACK_IP, COL_ATO])
    df[COL_IP] = "sintetico"
    return df


# --------------------------------------------------------------------------------------
# 2. C1 - Ingesta y feature engineering (solo usa el pasado de cada usuario)
# --------------------------------------------------------------------------------------
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Construye x_i (R^8), las reglas r1..r5 y c_i. El historial de cada usuario se forma
    SOLO con sus inicios de sesión exitosos anteriores al evento, de modo que no hay
    fuga temporal ni se usan las etiquetas de ataque como entrada.
    """
    d = df.sort_values(COL_TS, kind="mergesort").reset_index(drop=True).copy()
    d["anomalo"] = ((d[COL_ATTACK_IP] == 1) | (d[COL_ATO] == 1)).astype(int)
    d["dispositivo"] = (d[COL_BROWSER].astype(str) + "|" + d[COL_OS].astype(str) + "|"
                        + d[COL_DEVICE].astype(str))
    succ = d[COL_SUCCESS].astype(bool).astype(int)
    d["_succ"] = succ

    hour = d[COL_TS].dt.hour + d[COL_TS].dt.minute / 60.0
    ang = 2 * np.pi * hour / 24.0
    d["hora_sin"], d["hora_cos"] = np.sin(ang), np.cos(ang)

    g = d.groupby(COL_USER)
    n_prev = g["_succ"].cumsum() - succ                   # accesos exitosos previos
    d["n_prev"] = n_prev
    d["log_n_prev"] = np.log1p(n_prev)

    def prior_share(key: str) -> pd.Series:
        same = d.groupby([COL_USER, key])["_succ"].cumsum() - succ
        return np.where(n_prev > 0, same / n_prev.clip(lower=1), 0.0)

    d["frec_pais"] = prior_share(COL_COUNTRY)
    d["frec_asn"] = prior_share(COL_ASN)
    d["frec_dispositivo"] = prior_share("dispositivo")

    # Hora habitual: media circular de los accesos exitosos previos
    s_cum = (d["hora_sin"] * succ).groupby(d[COL_USER]).cumsum() - d["hora_sin"] * succ
    c_cum = (d["hora_cos"] * succ).groupby(d[COL_USER]).cumsum() - d["hora_cos"] * succ
    mean_ang = np.arctan2(s_cum, c_cum)
    diff = np.abs(np.angle(np.exp(1j * (ang - mean_ang))))     # 0..pi
    desv_h = diff * 24 / (2 * np.pi)                            # 0..12 h
    d["desv_horas"] = np.where(n_prev > 0, desv_h, np.nan)
    d["desv_horaria"] = np.where(n_prev > 0, desv_h / 12.0, 0.5)

    d["log_rtt"] = np.log1p(d[COL_RTT].astype(float).fillna(d[COL_RTT].median()))

    # Último acceso exitoso previo (para viaje imposible)
    d["_ctry_s"] = d[COL_COUNTRY].where(succ == 1)
    d["_ts_s"] = d[COL_TS].where(succ == 1)
    # Vectorizado (sin lambda por usuario) para soportar millones de filas
    d["prev_pais"] = d.groupby(COL_USER)["_ctry_s"].shift(1).groupby(d[COL_USER]).ffill()
    d["prev_ts"] = d.groupby(COL_USER)["_ts_s"].shift(1).groupby(d[COL_USER]).ffill()
    horas_desde_prev = (d[COL_TS] - d["prev_ts"]).dt.total_seconds() / 3600.0

    hist = n_prev >= 1
    d["r1_dispositivo_nuevo"] = (hist & (d["frec_dispositivo"] == 0)).astype(int)
    d["r2_pais_poco_habitual"] = (hist & (d["frec_pais"] < 0.2)).astype(int)
    d["r3_viaje_imposible"] = (d["prev_pais"].notna() & (d["prev_pais"] != d[COL_COUNTRY])
                               & (horas_desde_prev < 2)).astype(int)
    d["r4_horario_atipico"] = ((n_prev >= 5) & (d["desv_horas"] > 4)).astype(int)
    d["r5_asn_nuevo"] = (hist & (d["frec_asn"] == 0)).astype(int)
    d["c_i"] = sum(RULE_WEIGHTS[r] * d[r] for r in RULES)
    d["arranque_frio"] = (n_prev == 0).astype(int)

    # ---- Contexto global (v4), causal: cada evento solo ve eventos anteriores ----
    dev = d[COL_DEVICE].astype(str).str.lower()
    d["es_movil"] = (dev == "mobile").astype(int)
    d["es_desktop"] = (dev == "desktop").astype(int)
    fail = 1 - succ
    g_n = np.arange(len(d))
    g_f = fail.cumsum().to_numpy() - fail.to_numpy()
    p0 = (g_f + 1.0) / (g_n + 4.0)                      # tasa global de fallo previa (suavizada)
    m = 20.0                                            # fuerza del suavizado bayesiano
    for key, name in ((COL_ASN, "tasa_fallo_asn"), (COL_COUNTRY, "tasa_fallo_pais")):
        n_k = d.groupby(key).cumcount().to_numpy()
        f_k = fail.groupby(d[key]).cumsum().to_numpy() - fail.to_numpy()
        d[name] = (f_k + m * p0) / (n_k + m)
    d["log_vol_asn"] = np.log1p(d.groupby(COL_ASN).cumcount())

    return d.drop(columns=["_succ", "_ctry_s", "_ts_s"])


def diagnose(raw: pd.DataFrame, d: pd.DataFrame) -> dict:
    """
    Análisis exploratorio orientado a la detección: ¿qué variables separan de verdad los
    accesos anómalos de los legítimos? (Etapa 1 del procedimiento metodológico).
    """
    from sklearn.metrics import roc_auc_score
    y = d["anomalo"].to_numpy()
    lab = np.where(y == 1, "anomalo", "legitimo")
    per_user = d.groupby(COL_USER).size()

    def auc(col, mask=None):
        m = np.ones(len(d), bool) if mask is None else mask
        yy, xx = y[m], d.loc[m, col].astype(float).fillna(0).to_numpy()
        return round(float(roc_auc_score(yy, xx)), 3) if 0 < yy.sum() < len(yy) else None

    hist = (d["n_prev"] >= 1).to_numpy()
    cols = FEATURES + FEATURES_GLOBAL + RULES + ["c_i", COL_RTT]

    def top_share(col, k=5):
        out = {}
        for name in ("legitimo", "anomalo"):
            s = d.loc[lab == name, col].astype(str).value_counts(normalize=True).head(k)
            out[name] = {i: round(float(v), 3) for i, v in s.items()}
        return out

    legit_asn = set(d.loc[y == 0, COL_ASN])
    legit_users = set(d.loc[(y == 0) & d[COL_SUCCESS].astype(bool), COL_USER])
    atk = d[y == 1]
    return {
        "archivo_ordenado_por_fecha": bool(raw[COL_TS].is_monotonic_increasing),
        "rango_fechas": [str(d[COL_TS].min()), str(d[COL_TS].max())],
        "eventos": int(len(d)), "usuarios": int(per_user.size),
        "eventos_por_usuario_p50_p90_max": [int(per_user.quantile(.5)), int(per_user.quantile(.9)),
                                            int(per_user.max())],
        "tasa_anomalos": round(float(y.mean()), 4), "n_ato": int(d[COL_ATO].sum()),
        "exito_login": {k: round(float(d.loc[lab == k, COL_SUCCESS].astype(bool).mean()), 4)
                        for k in ("legitimo", "anomalo")},
        "primer_acceso": {k: round(float((d.loc[lab == k, "n_prev"] == 0).mean()), 4)
                          for k in ("legitimo", "anomalo")},
        "anomalos_sobre_usuarios_con_historial_legitimo": round(float(atk[COL_USER].isin(legit_users).mean()), 4),
        "anomalos_con_asn_visto_en_legitimos": round(float(atk[COL_ASN].isin(legit_asn).mean()), 4),
        "pais_top5": top_share(COL_COUNTRY), "tipo_dispositivo_top5": top_share(COL_DEVICE),
        "asn_top5": top_share(COL_ASN),
        "auc_univariada_todos": {c: auc(c) for c in cols},
        "auc_univariada_con_historial": {c: auc(c, hist) for c in cols},
        "tasa_reglas": {r: {k: round(float(d.loc[lab == k, r].mean()), 4) for k in ("legitimo", "anomalo")}
                        for r in RULES},
        "nota_auc": "0.5 = no separa; > 0.5 = valores altos indican anomalía; < 0.5 = relación invertida",
    }


def temporal_split(d: pd.DataFrame, split=(0.70, 0.15, 0.15)):
    n = len(d)
    a, b = int(n * split[0]), int(n * (split[0] + split[1]))
    return d.iloc[:a].copy(), d.iloc[a:b].copy(), d.iloc[b:].copy()


# --------------------------------------------------------------------------------------
# 3. C2 / C3 / C4 - detección, consolidación y decisión
# --------------------------------------------------------------------------------------
def train_iforest(train: pd.DataFrame, cfg: Config, features=None) -> IsolationForest:
    features = features or FEATURES
    model = IsolationForest(n_estimators=cfg.n_estimators, max_samples=cfg.max_samples,
                            contamination="auto", random_state=cfg.seed, n_jobs=-1)
    model.fit(train[features].to_numpy())
    return model


def anomaly_score(model: IsolationForest, d: pd.DataFrame, features=None) -> np.ndarray:
    """Ecuación 1: s(x, n) = 2^(-E[h(x)]/c(n)). En scikit-learn, score_samples = -s(x, n)."""
    return -model.score_samples(d[features or FEATURES].to_numpy())


def rescale(s: np.ndarray, s_ref: float) -> np.ndarray:
    """Ecuación 1b: s'(x) = max(0, (s - s_ref) / (1 - s_ref))."""
    return np.clip((s - s_ref) / (1.0 - s_ref), 0.0, 1.0)


def risk_score(s_prime: np.ndarray, c: np.ndarray, alpha: float) -> np.ndarray:
    """Ecuación 2: R_i = alpha * s'(x_i) + (1 - alpha) * c_i."""
    return alpha * s_prime + (1.0 - alpha) * c


def decide(R: np.ndarray, theta1: float, theta2: float, cold: np.ndarray | None = None,
           cold_start_otp: bool = True) -> np.ndarray:
    """Reglas P2, P3 y P5. 0 = acceso directo, 1 = OTP, 2 = bloqueo."""
    dec = np.where(R > theta2, 2, np.where(R > theta1, 1, 0))
    if cold_start_otp and cold is not None:
        dec = np.where((cold == 1) & (dec == 0), 1, dec)
    return dec


# --------------------------------------------------------------------------------------
# 4. Métricas (dos análisis binarios sobre la decisión de tres niveles)
# --------------------------------------------------------------------------------------
def binary_metrics(y: np.ndarray, pred: np.ndarray) -> dict:
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    fpr = fp / (fp + tn) if fp + tn else 0.0
    fnr = fn / (fn + tp) if fn + tp else 0.0
    denom = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn) - (fp * fn)) / denom if denom else 0.0
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, precision=precision, recall=recall, f1=f1,
                fpr=fpr, fnr=fnr, mcc=mcc)


def evaluate_decisions(y: np.ndarray, dec: np.ndarray) -> dict:
    """Análisis theta1: positivo = OTP o bloqueo. Análisis theta2: positivo = bloqueo."""
    return {"analisis_theta1": binary_metrics(y, (dec >= 1).astype(int)),
            "analisis_theta2": binary_metrics(y, (dec == 2).astype(int))}


def confusion_3x2(y: np.ndarray, dec: np.ndarray) -> pd.DataFrame:
    tab = pd.crosstab(pd.Series(dec).map(DECISION_NAMES).rename("decision"),
                      pd.Series(y).map({0: "legitimo", 1: "anomalo"}).rename("etiqueta_real"))
    return tab.reindex(index=list(DECISION_NAMES.values()), columns=["legitimo", "anomalo"],
                       fill_value=0)


# --------------------------------------------------------------------------------------
# 5. Calibración en validación (el conjunto de prueba no se usa aquí)
# --------------------------------------------------------------------------------------
def calibrate(val: pd.DataFrame, s_prime: np.ndarray, cfg: Config, alpha_grid) -> dict:
    """
    Búsqueda en malla: maximiza F1 del análisis theta1 sujeto a FPR <= max_fpr_theta1.
    theta2 es el menor umbral > theta1 cuyo FPR de bloqueo <= max_fpr_theta2.
    """
    y = val[cfg.label].to_numpy()
    cold = val["arranque_frio"].to_numpy()
    c = val["c_i"].to_numpy()
    best, fallback = None, None
    for alpha in alpha_grid:
        R = risk_score(s_prime, c, alpha)
        for t1 in cfg.theta_grid:
            dec = decide(R, t1, 1.01, cold, cfg.cold_start_otp)
            m = binary_metrics(y, (dec >= 1).astype(int))
            cand = dict(alpha=float(alpha), theta1=float(t1), f1_val=m["f1"], fpr_val=m["fpr"])
            if m["fpr"] <= cfg.max_fpr_theta1 and (best is None or m["f1"] > best["f1_val"]):
                best = cand
            # respaldo: menor FPR posible y, a igualdad, mayor F1
            if fallback is None or (m["fpr"], -m["f1"]) < (fallback["fpr_val"], -fallback["f1_val"]):
                fallback = cand
    if best is None:   # ninguna combinación cumple FPR <= umbral: se declara y se usa el respaldo
        best = dict(fallback, restriccion_fpr_incumplida=True)
    R = risk_score(s_prime, c, best["alpha"])
    best["theta2"] = 0.95
    for t2 in cfg.theta_grid:
        if t2 <= best["theta1"]:
            continue
        m = binary_metrics(y, (R > t2).astype(int))
        if m["fpr"] <= cfg.max_fpr_theta2:
            best["theta2"] = float(t2)
            break
    return best


# --------------------------------------------------------------------------------------
# 6. Prueba pareada por bloques temporales (baseline vs AdaptAuth-ML)
# --------------------------------------------------------------------------------------
def paired_block_test(test: pd.DataFrame, dec_a: np.ndarray, dec_b: np.ndarray,
                      y_col: str, n_blocks: int) -> dict:
    """
    Divide el conjunto de prueba en n bloques temporales consecutivos y compara el F1
    (análisis theta1) de ambas condiciones sobre los MISMOS bloques: muestras pareadas.
    """
    y = test[y_col].to_numpy()
    idx = np.array_split(np.arange(len(test)), n_blocks)
    f1_a = np.array([binary_metrics(y[i], (dec_a[i] >= 1).astype(int))["f1"] for i in idx])
    f1_b = np.array([binary_metrics(y[i], (dec_b[i] >= 1).astype(int))["f1"] for i in idx])
    diff = f1_b - f1_a
    shapiro_p = float(stats.shapiro(diff).pvalue) if np.ptp(diff) > 0 else 1.0
    if shapiro_p > 0.05:
        t = stats.ttest_rel(f1_b, f1_a)
        test_name, stat, p = "t de Student pareada", float(t.statistic), float(t.pvalue)
    else:
        w = stats.wilcoxon(f1_b, f1_a)
        test_name, stat, p = "Wilcoxon de rangos con signo", float(w.statistic), float(w.pvalue)
    sd = diff.std(ddof=1)
    dz = float(diff.mean() / sd) if sd > 0 else float("inf")
    return dict(n_bloques=n_blocks, f1_baseline=f1_a.tolist(), f1_adaptauth=f1_b.tolist(),
                media_baseline=float(f1_a.mean()), media_adaptauth=float(f1_b.mean()),
                de_baseline=float(f1_a.std(ddof=1)), de_adaptauth=float(f1_b.std(ddof=1)),
                shapiro_p_diferencias=shapiro_p, prueba=test_name, estadistico=stat, p_valor=p,
                d_cohen_dz=dz)


# --------------------------------------------------------------------------------------
# 7. Latencia (RNF03 / CA-C2-03)
# --------------------------------------------------------------------------------------
def inference_latency_ms(model, d: pd.DataFrame, n: int = 300, seed: int = 0, features=None) -> dict:
    rng = np.random.default_rng(seed)
    X = d[features or FEATURES].to_numpy()
    score = model.score_samples if hasattr(model, "score_samples") else model.predict_proba
    times = []
    for i in rng.integers(0, len(X), n):
        t0 = time.perf_counter()
        score(X[i:i + 1])
        times.append((time.perf_counter() - t0) * 1000)
    t = np.array(times)
    return dict(p50_ms=float(np.percentile(t, 50)), p95_ms=float(np.percentile(t, 95)),
                p99_ms=float(np.percentile(t, 99)), media_ms=float(t.mean()))


def to_json(obj, path):
    def conv(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, Config):
            return asdict(o)
        raise TypeError(type(o))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=conv)


In [ ]:
%%writefile figures.py
"""Figuras del experimento base (paleta de referencia: azul y naranja, marcas finas)."""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11, "axes.edgecolor": GRID,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150,
})


def _grid(ax, axis="y"):
    ax.grid(axis=axis, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def risk_distribution(R, y, t1, t2, path):
    fig, ax = plt.subplots(figsize=(8, 4.2))
    bins = np.linspace(0, 1, 41)
    ax.hist(R[y == 0], bins=bins, density=True, histtype="step", linewidth=2, color=BLUE,
            label="Accesos legítimos")
    ax.hist(R[y == 1], bins=bins, density=True, histtype="step", linewidth=2, color=ORANGE,
            label="Accesos anómalos")
    ax.set_yscale("log")
    top = ax.get_ylim()[1]
    for t, name, f, ha, dx in ((t1, "θ1", 0.5, "right", -0.01), (t2, "θ2", 0.12, "left", 0.01)):
        ax.axvline(t, color=INK2, linewidth=1, linestyle="--")
        ax.text(t + dx, top * f, f"{name} = {t:.2f}", color=INK2, fontsize=10, ha=ha)
    ax.set_xlabel("Score de riesgo consolidado R_i")
    ax.set_ylabel("Densidad (escala log)")
    _grid(ax)
    ax.legend(frameon=False, loc="upper right")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def paired_blocks(res, path):
    a, b = np.array(res["f1_baseline"]), np.array(res["f1_adaptauth"])
    x = np.arange(1, len(a) + 1)
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.plot(x, a, color=BLUE, linewidth=2, marker="o", markersize=7, label="Baseline (solo reglas)")
    ax.plot(x, b, color=ORANGE, linewidth=2, marker="o", markersize=7, label="AdaptAuth-ML")
    ax.set_xticks(x)
    ax.set_xlabel("Bloque temporal del conjunto de prueba")
    ax.set_ylabel("F1 (análisis θ1)")
    ax.set_ylim(0, 1)
    _grid(ax)
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def shap_bars(mean_abs, path):
    s = mean_abs.sort_values()
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.barh(s.index, s.values, color=BLUE, height=0.6)
    for i, v in enumerate(s.values):
        ax.text(v, i, f" {v:.3f}", va="center", color=INK2, fontsize=9)
    ax.set_xlabel("Importancia media |SHAP| sobre el score de anomalía")
    _grid(ax, "x")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def model_comparison(tabla, path):
    """Barras horizontales agrupadas: AUC-ROC y F1 (θ1) por modelo, misma escala 0 a 1."""
    t = tabla.iloc[::-1].reset_index(drop=True)
    y = np.arange(len(t))
    h = 0.36
    fig, ax = plt.subplots(figsize=(8.5, 0.6 * len(t) + 1.6))
    ax.barh(y + h / 2 + 0.01, t["auc_roc"], height=h, color=BLUE, label="AUC-ROC")
    ax.barh(y - h / 2 - 0.01, t["f1_theta1"], height=h, color=ORANGE, label="F1 (análisis θ1)")
    for i, (a, f) in enumerate(zip(t["auc_roc"], t["f1_theta1"])):
        ax.text(a + 0.01, i + h / 2, f"{a:.2f}", va="center", color=INK2, fontsize=9)
        ax.text(f + 0.01, i - h / 2, f"{f:.2f}", va="center", color=INK2, fontsize=9)
    ax.axvline(0.5, color=INK2, linewidth=1, linestyle="--")
    ax.text(0.505, -0.75, "azar (AUC 0.5)", color=INK2, fontsize=9)
    ax.set_yticks(y)
    ax.set_yticklabels(t["modelo"].str.replace("_", " "))
    ax.set_xlim(0, 1.08)
    ax.set_ylim(-0.9, len(t) - 0.4)
    _grid(ax, "x")
    ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


In [ ]:
%%writefile run_experimento.py
"""
Experimento v4 de AdaptAuth-ML: corrección basada en el diagnóstico del RBA real.

Cambios respecto a v3 (run_baseline.py):
  - Población evaluada: inicios de sesión con contraseña correcta (punto donde actúa la
    autenticación basada en riesgo). Los intentos fallidos se usan solo como contexto.
  - Variables de contexto global sin etiquetas (FEATURES_GLOBAL): tipo de dispositivo y tasa
    previa de fallos / volumen del ASN y del país.
  - Regla P5 (OTP a todo primer acceso) desactivada por defecto: con mediana de 2 accesos por
    usuario generaba por sí sola la mayor parte del FPR. Se reporta también con P5 activa.
  - Comparación en una sola corrida:
        reglas_estaticas        baseline de la Alternativa 1 (solo c_i)
        iforest_usuario         diseño v3 (x_i en R^8)
        iforest_usuario_global  x_i + contexto global
        adaptauth_v4            alpha * s'(IF usuario+global) + (1 - alpha) * c_i   <- propuesta
        referencia_supervisada  HistGradientBoosting con etiquetas (cota superior, no es la propuesta)

Uso:
    python run_experimento.py --data ruta/rba_muestra_usuarios_5pct.parquet
    python run_experimento.py --synthetic
"""
import argparse
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

sys.path.insert(0, str(Path(__file__).parent / "src"))
import adaptauth as aa  # noqa: E402
import figures  # noqa: E402

OUT = Path(__file__).parent / "results"
F_USR = aa.FEATURES
F_ALL = aa.FEATURES + aa.FEATURES_GLOBAL


def evaluar(nombre, s_val, s_test, val, test, cfg, alpha_grid=(1.0,), use_c=False):
    """Calibra en validación (máx. F1 con FPR <= 5 %) y evalúa en prueba."""
    v = val if use_c else val.assign(c_i=0.0)
    cal = aa.calibrate(v, s_val, cfg, alpha_grid)
    c = test["c_i"].to_numpy() if use_c else np.zeros(len(test))
    R = aa.risk_score(s_test, c, cal["alpha"])
    dec = aa.decide(R, cal["theta1"], cal["theta2"], test["arranque_frio"].to_numpy(), cfg.cold_start_otp)
    y = test[cfg.label].to_numpy()
    res = dict(calibracion=cal, **aa.evaluate_decisions(y, dec),
               auc_roc=float(roc_auc_score(y, R)), auc_pr=float(average_precision_score(y, R)),
               score_medio_legitimos=float(R[y == 0].mean()))
    print(f"  {nombre:<24} AUC-ROC={res['auc_roc']:.3f} AUC-PR={res['auc_pr']:.3f} | "
          f"θ1: F1={res['analisis_theta1']['f1']:.3f} FPR={res['analisis_theta1']['fpr']:.3f} | "
          f"θ2: F1={res['analisis_theta2']['f1']:.3f} FPR={res['analisis_theta2']['fpr']:.4f}"
          + ("  [FPR<=5% no alcanzable]" if cal.get("restriccion_fpr_incumplida") else ""))
    return res, dec, R


def main():
    ap = argparse.ArgumentParser()
    g = ap.add_mutually_exclusive_group(required=True)
    g.add_argument("--data")
    g.add_argument("--synthetic", action="store_true")
    ap.add_argument("--user-pct", type=float, default=None)
    ap.add_argument("--todos-los-intentos", action="store_true",
                    help="evaluar también intentos con contraseña incorrecta")
    ap.add_argument("--no-shap", action="store_true")
    args = ap.parse_args()

    cfg = aa.Config(cold_start_otp=False)
    OUT.mkdir(exist_ok=True); (OUT / "figuras").mkdir(exist_ok=True)
    t0 = time.time()
    raw = aa.generate_synthetic_rba(seed=cfg.seed) if args.synthetic else aa.load_rba(args.data, user_pct=args.user_pct)
    d = aa.build_features(raw)                      # el contexto se calcula con TODOS los intentos
    diag = aa.diagnose(raw, d)
    aa.to_json(diag, OUT / "diagnostico.json")
    poblacion = d if args.todos_los_intentos else d[d[aa.COL_SUCCESS].astype(bool)].reset_index(drop=True)
    train, val, test = aa.temporal_split(poblacion, cfg.split)
    print(f"[datos] {len(d):,} intentos | población evaluada {len(poblacion):,} "
          f"({'todos' if args.todos_los_intentos else 'contraseña correcta'}) | "
          f"anómalos en prueba {int(test['anomalo'].sum()):,} ({test['anomalo'].mean():.1%}) | {time.time()-t0:.0f}s")
    if val["anomalo"].sum() == 0 or test["anomalo"].sum() == 0:
        sys.exit("ERROR: validación o prueba sin eventos anómalos.")

    y_val = val[cfg.label].to_numpy() == 0
    res, decs, Rs = {}, {}, {}
    print("[modelos] calibrados en validación, evaluados en prueba:")

    # B0. Reglas estáticas (Alternativa 1 del PI6)
    res["reglas_estaticas"], decs["reglas_estaticas"], Rs["reglas_estaticas"] = evaluar(
        "reglas_estaticas", np.zeros(len(val)), np.zeros(len(test)), val, test, cfg, (0.0,), use_c=True)

    # M1 / M2. Isolation Forest con y sin contexto global
    modelos = {}
    for nombre, feats in (("iforest_usuario", F_USR), ("iforest_usuario_global", F_ALL)):
        m = aa.train_iforest(train, cfg, feats)
        s_v, s_t = aa.anomaly_score(m, val, feats), aa.anomaly_score(m, test, feats)
        s_ref = float(np.median(s_v[y_val]))
        sp_v, sp_t = aa.rescale(s_v, s_ref), aa.rescale(s_t, s_ref)
        res[nombre], decs[nombre], Rs[nombre] = evaluar(nombre, sp_v, sp_t, val, test, cfg)
        res[nombre]["s_ref"] = s_ref
        modelos[nombre] = (m, feats, sp_v, sp_t)

    # M3. AdaptAuth-ML v4 (propuesta): IF usuario+global combinado con c_i
    _, _, sp_v, sp_t = modelos["iforest_usuario_global"]
    res["adaptauth_v4"], decs["adaptauth_v4"], Rs["adaptauth_v4"] = evaluar(
        "adaptauth_v4", sp_v, sp_t, val, test, cfg, cfg.alpha_grid, use_c=True)
    cfg_p5 = aa.Config(cold_start_otp=True)
    res["adaptauth_v4_con_P5"], _, _ = evaluar(
        "adaptauth_v4_con_P5", sp_v, sp_t, val, test, cfg_p5, cfg.alpha_grid, use_c=True)

    # R1. Referencia supervisada (cota superior con las mismas variables)
    hgb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=cfg.seed)
    hgb.fit(train[F_ALL].to_numpy(), train[cfg.label].to_numpy())
    p_v = hgb.predict_proba(val[F_ALL].to_numpy())[:, 1]
    p_t = hgb.predict_proba(test[F_ALL].to_numpy())[:, 1]
    res["referencia_supervisada"], decs["referencia_supervisada"], Rs["referencia_supervisada"] = evaluar(
        "referencia_supervisada", p_v, p_t, val, test, cfg)

    # Tabla comparativa
    filas = []
    for k, r in res.items():
        filas.append(dict(modelo=k, auc_roc=r["auc_roc"], auc_pr=r["auc_pr"],
                          f1_theta1=r["analisis_theta1"]["f1"], fpr_theta1=r["analisis_theta1"]["fpr"],
                          recall_theta1=r["analisis_theta1"]["recall"], precision_theta1=r["analisis_theta1"]["precision"],
                          f1_theta2=r["analisis_theta2"]["f1"], fpr_theta2=r["analisis_theta2"]["fpr"],
                          mcc_theta1=r["analisis_theta1"]["mcc"], alpha=r["calibracion"]["alpha"],
                          theta1=r["calibracion"]["theta1"], theta2=r["calibracion"]["theta2"],
                          fpr_5pct_alcanzable=not r["calibracion"].get("restriccion_fpr_incumplida", False)))
    tabla = pd.DataFrame(filas)
    tabla.to_csv(OUT / "tabla_comparativa.csv", index=False)

    y = test[cfg.label].to_numpy()
    resultados = dict(
        origen_datos="SINTETICO (solo prueba)" if args.synthetic else f"RBA real: {args.data}",
        poblacion="todos los intentos" if args.todos_los_intentos else "inicios con contraseña correcta",
        volumen=dict(intentos_totales=len(d), poblacion=len(poblacion), train=len(train), val=len(val),
                     test=len(test), anomalos_test=int(y.sum()), ato_test=int(test[aa.COL_ATO].sum()),
                     primer_acceso_test=int(test["arranque_frio"].sum())),
        modelos=res,
        prueba_pareada_vs_reglas=aa.paired_block_test(test, decs["reglas_estaticas"], decs["adaptauth_v4"],
                                                      cfg.label, cfg.n_test_blocks),
        prueba_pareada_vs_v3=aa.paired_block_test(test, decs["iforest_usuario"], decs["adaptauth_v4"],
                                                  cfg.label, cfg.n_test_blocks),
        latencia_inferencia_iforest=aa.inference_latency_ms(modelos["iforest_usuario_global"][0], test,
                                                            features=F_ALL),
    )
    aa.confusion_3x2(y, decs["adaptauth_v4"]).to_csv(OUT / "matriz_3x2_adaptauth_v4.csv")

    if not args.no_shap:
        import shap
        m, feats = modelos["iforest_usuario_global"][:2]
        ex = shap.TreeExplainer(m)
        rng = np.random.default_rng(cfg.seed)
        imp = []
        for b in np.array_split(np.arange(len(test)), 5):
            sel = rng.choice(b, size=min(400, len(b)), replace=False)
            imp.append(np.abs(ex.shap_values(test.iloc[sel][feats].to_numpy())).mean(axis=0))
        imp = pd.DataFrame(imp, columns=feats)
        top_ref = set(imp.mean().sort_values(ascending=False).index[:3])
        coinc = sum(set(r.sort_values(ascending=False).index[:3]) == top_ref for _, r in imp.iterrows())
        resultados["shap_iforest_usuario_global"] = dict(
            top3=list(imp.mean().sort_values(ascending=False).index[:3]), bloques_que_coinciden=coinc,
            cumple=coinc >= 4, media_abs=imp.mean().round(4).to_dict())
        figures.shap_bars(imp.mean(), OUT / "figuras" / "shap_importancia.png")
        # importancia en la referencia supervisada (qué variables separan de verdad los ataques)
        from sklearn.inspection import permutation_importance
        sel = rng.choice(len(test), size=min(20000, len(test)), replace=False)
        pi = permutation_importance(hgb, test.iloc[sel][F_ALL].to_numpy(), y[sel], scoring="average_precision",
                                    n_repeats=3, random_state=cfg.seed)
        resultados["importancia_referencia_supervisada"] = dict(
            sorted(zip(F_ALL, np.round(pi.importances_mean, 4)), key=lambda kv: -kv[1]))

    c = res["adaptauth_v4"]["calibracion"]
    figures.risk_distribution(Rs["adaptauth_v4"], y, c["theta1"], c["theta2"], OUT / "figuras" / "distribucion_riesgo.png")
    figures.paired_blocks(resultados["prueba_pareada_vs_reglas"], OUT / "figuras" / "f1_por_bloque.png")
    figures.model_comparison(tabla, OUT / "figuras" / "comparacion_modelos.png")
    aa.to_json(resultados, OUT / "metricas_v4.json")
    print(tabla[["modelo", "auc_roc", "auc_pr", "f1_theta1", "fpr_theta1", "fpr_5pct_alcanzable"]].round(3).to_string(index=False))
    print(f"[listo] {time.time()-t0:.0f}s -> {OUT}")


if __name__ == "__main__":
    main()


## 2. Dataset desde Google Drive

In [ ]:
USER_PCT = 5      # % de usuarios. Si cambias este valor se crea una muestra nueva.
RUTA_ZIP = ""     # opcional: ruta exacta del zip en tu Drive. Si la dejas vacía, se busca sola.

import os
from google.colab import drive
drive.mount("/content/drive")
RAIZ = "/content/drive/MyDrive"
CARPETA = f"{RAIZ}/AdaptAuth-ML"
os.makedirs(CARPETA, exist_ok=True)
MUESTRA = f"{CARPETA}/rba_muestra_usuarios_{USER_PCT}pct.parquet"

if os.path.exists(MUESTRA):
    print("Muestra encontrada en Drive, no se lee el zip:", MUESTRA)
else:
    if not RUTA_ZIP:   # busca en todo tu Drive cualquier .zip o .csv con "rba" en el nombre
        candidatos = [os.path.join(r, f) for r, _, fs in os.walk(RAIZ) for f in fs
                      if "rba" in f.lower() and f.lower().endswith((".zip", ".csv"))]
        for c in candidatos:
            print(f"  encontrado: {c} ({os.path.getsize(c) / 1e9:.2f} GB)")
        if candidatos:
            RUTA_ZIP = max(candidatos, key=os.path.getsize)
        else:
            print("No está en tu Drive: descargando de Zenodo (solo esta vez)...")
            RUTA_ZIP = f"{CARPETA}/rba-dataset.zip"
            !wget -q --show-progress -O "{RUTA_ZIP}" "https://zenodo.org/records/6782156/files/rba-dataset.zip?download=1"
    print("Usando:", RUTA_ZIP)
    print("Creando la muestra por usuario (solo esta vez, tarda unos minutos)...")
    import adaptauth as aa
    aa.load_rba(RUTA_ZIP, user_pct=USER_PCT).to_parquet(MUESTRA, index=False)
    print("Muestra guardada en Drive:", MUESTRA)
!ls -lh "{CARPETA}"

## 3. Ejecutar el experimento comparativo

In [ ]:
!python run_experimento.py --data "{MUESTRA}"

## 4. Resultados

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
r = json.load(open("results/metricas_v4.json"))
print("Población:", r["poblacion"]); print("Volumen:", r["volumen"])
display(pd.read_csv("results/tabla_comparativa.csv").round(4))
for k in ["prueba_pareada_vs_reglas", "prueba_pareada_vs_v3"]:
    p = r[k]
    print(f"{k}: {p['prueba']}, p={p['p_valor']:.4f}, dz={p['d_cohen_dz']:.2f} | M_A={p['media_baseline']:.3f} M_B={p['media_adaptauth']:.3f}")
print("Latencia IF (ms):", {k: round(v, 2) for k, v in r["latencia_inferencia_iforest"].items()})
print("SHAP IF usuario+global:", r.get("shap_iforest_usuario_global", {}).get("top3"), "| estable:", r.get("shap_iforest_usuario_global", {}).get("cumple"))
print("Importancia (referencia supervisada):", r.get("importancia_referencia_supervisada"))
print(pd.read_csv("results/matriz_3x2_adaptauth_v4.csv", index_col=0))
for f in ["comparacion_modelos", "distribucion_riesgo", "f1_por_bloque", "shap_importancia"]:
    display(Image(f"results/figuras/{f}.png"))

## 5. Guardar en Google Drive
Quedan en **Mi unidad → AdaptAuth-ML → results_v4**. Mándame `metricas_v4.json` y `tabla_comparativa.csv`.

In [ ]:
!rm -rf "{CARPETA}/results_v4" && cp -r results "{CARPETA}/results_v4"
print("Resultados guardados en Mi unidad / AdaptAuth-ML / results_v4")